# CELL MESH Demo: HNSC 200-cell h5ad Workflow

This notebook demonstrates the main CELL MESH analysis flow using `cellmesh/data/demo_HNSC_200cell.h5ad`:

1. Load an h5ad single-cell dataset.
2. Load packaged enzyme and sensor priors.
3. Filter priors to genes present in the h5ad file and select a small reproducible demo subset.
4. Run `run_cell_mesh()`.
5. Inspect communication events and draw a sender-receiver heatmap for one metabolite.


## 1. Imports

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cellmesh
from cellmesh import read_anndata, load_cell_mesh_database, run_cell_mesh

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)

print("cellmesh", cellmesh.__version__)
print("cellmesh path:", cellmesh.__file__)


## 2. Load the Demo h5ad File

The demo uses `Celltype (malignancy)` as the cell-group column because all three groups have enough cells in this 200-cell subset.

In [ ]:
PACKAGE_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
h5ad_path = PACKAGE_ROOT / "cellmesh" / "data" / "demo_HNSC_200cell.h5ad"

adata = read_anndata(h5ad_path, mode="h5ad")
cell_type_key = "Celltype (malignancy)"

print("h5ad:", h5ad_path)
print("shape:", adata.shape)
print("obs columns:", list(adata.obs.columns))
display(adata.obs[cell_type_key].value_counts().to_frame("n_cells"))


## 3. Load and Filter Packaged Priors

`load_cell_mesh_database()` loads the packaged versioned priors. For a quick demo, the priors are filtered to genes present in the h5ad file and then restricted to the first 20 metabolite/HMDB pairs that have both enzyme and sensor evidence.

In [ ]:
enzyme_metabolite, metabolite_sensor = load_cell_mesh_database()
genes_present = set(pd.Index(adata.var_names).astype(str))

enzyme_present = enzyme_metabolite[enzyme_metabolite["gene"].astype(str).isin(genes_present)].copy()
sensor_present = metabolite_sensor[metabolite_sensor["sensor_gene"].astype(str).isin(genes_present)].copy()

matched_pairs = sorted(
    set(zip(enzyme_present["metabolite"], enzyme_present["hmdb_id"]))
    .intersection(set(zip(sensor_present["metabolite"], sensor_present["hmdb_id"])))
)

MAX_METABOLITES = 20
selected_pairs = pd.DataFrame(matched_pairs[:MAX_METABOLITES], columns=["metabolite", "hmdb_id"])
enzyme_demo = enzyme_present.merge(selected_pairs, on=["metabolite", "hmdb_id"])
sensor_demo = sensor_present.merge(selected_pairs, on=["metabolite", "hmdb_id"])

print("loaded enzyme prior:", enzyme_metabolite.shape)
print("loaded sensor prior:", metabolite_sensor.shape)
print("enzyme rows with expressed genes:", enzyme_present.shape)
print("sensor rows with expressed genes:", sensor_present.shape)
print("matched metabolite/HMDB pairs:", len(matched_pairs))
print("demo enzyme prior:", enzyme_demo.shape)
print("demo sensor prior:", sensor_demo.shape)
display(selected_pairs.head(20))


## 4. Run CELL MESH

This demo uses `min_cells=10` because the h5ad file contains 200 cells. The public default remains 100, which is appropriate for larger datasets. `n_perms=100` is used so empirical permutation p-values and FDR are available in the event table.


In [ ]:
res = run_cell_mesh(
    adata,
    enzyme_metabolite=enzyme_demo,
    metabolite_sensor=sensor_demo,
    cell_type_key=cell_type_key,
    min_cells=10,
    min_expr_frac=0.05,
    allow_self=True,
    n_perms=100,
    random_state=0,
)

print("events:", res.events.shape)
print("sender_scores:", res.sender_scores.shape)
print("receiver_scores:", res.receiver_scores.shape)
print("availability result keys:", sorted(res.availability_results.keys()))
print("events with permutation p-value:", res.events["perm_pvalue"].notna().sum())
print("events with FDR:", res.events["fdr"].notna().sum())


## 5. Inspect Top Events

In [ ]:
event_cols = [
    "sender",
    "receiver",
    "metabolite",
    "hmdb_id",
    "sensor_gene",
    "sensor_type",
    "metabolite_availability",
    "sensor_score",
    "cell_mesh_score",
    "perm_pvalue",
    "fdr",
    "confidence_tier",
]

display(res.events[event_cols].head(20))


In [ ]:
len(res.events[event_cols])

## 6. Sender-side Metabolite Availability

`sender_scores` is a metabolite/HMDB by sender-group availability matrix.

In [ ]:
display(res.sender_scores.head(10).round(3))


## 7. Receiver-side Sensor Scores

`receiver_scores` stores the normalized sensor expression score for each metabolite-sensor-receiver combination.

In [ ]:
display(res.receiver_scores.head(20).round(3))


## 8. Heatmap: FDR for One HMDB ID and Sensor Gene

Specify `target_hmdb_id` and `target_sensor_gene`, then draw a sender × receiver heatmap for that event family. Color represents `-log10(FDR)` and the number inside each tile is `cell_mesh_score`.


In [ ]:
assert not res.events.empty

# Choose a specific HMDB ID and sensor gene to visualize.
# These values are present in the demo subset generated above.
target_hmdb_id = "HMDB0000374"
target_sensor_gene = "NR2F2"

target_events = res.events[
    (res.events["hmdb_id"] == target_hmdb_id)
    & (res.events["sensor_gene"] == target_sensor_gene)
].copy()

assert not target_events.empty, f"No events found for {target_hmdb_id} / {target_sensor_gene}"

target_metabolite = target_events["metabolite"].iloc[0]
score_matrix = target_events.pivot(index="sender", columns="receiver", values="cell_mesh_score")
fdr_matrix = target_events.pivot(index="sender", columns="receiver", values="fdr")

min_positive_fdr = 1.0 / (100 + 1)
log_fdr_matrix = -np.log10(fdr_matrix.clip(lower=min_positive_fdr))
plot_values = np.ma.masked_invalid(log_fdr_matrix.to_numpy(dtype=float))

fig, ax = plt.subplots(figsize=(1.4 * len(score_matrix.columns) + 3, 1.0 * len(score_matrix.index) + 2.5))
image = ax.imshow(plot_values, cmap="viridis", aspect="auto")

ax.set_xticks(np.arange(len(score_matrix.columns)))
ax.set_xticklabels(score_matrix.columns, rotation=35, ha="right")
ax.set_yticks(np.arange(len(score_matrix.index)))
ax.set_yticklabels(score_matrix.index)
ax.set_xlabel("Receiver")
ax.set_ylabel("Sender")
ax.set_title(f"{target_metabolite} ({target_hmdb_id}) / {target_sensor_gene}")

for i, sender in enumerate(score_matrix.index):
    for j, receiver in enumerate(score_matrix.columns):
        score = score_matrix.loc[sender, receiver]
        if pd.notna(score):
            ax.text(j, i, f"{score:.2f}", ha="center", va="center", color="white", fontsize=10)

cbar = fig.colorbar(image, ax=ax)
cbar.set_label("-log10(FDR)")
fig.tight_layout()
plt.show()

display(target_events[["sender", "receiver", "metabolite", "hmdb_id", "sensor_gene", "sensor_type", "cell_mesh_score", "perm_pvalue", "fdr"]])


## 9. Optional Next Step

For a full analysis, increase `MAX_METABOLITES` or pass the full packaged priors to `run_cell_mesh()`. For empirical p-values and FDR, set `n_perms` to a positive value.